# 字符语言模型：预测下一个字

为每个字符分配整数编号，用前文预测下一个字符。
默认只训练两步，验证 AdamW、梯度裁剪、随机续写与 KV cache，续写可能重复或跳跃。
请在 `examples` 目录运行，使用项目的 Python 环境。

In [1]:
from qutai import Stream, lm, nn, visualization

# 使用 CPU 时改为 "cpu"；模型与浮点输入保持相同设备和精度。
device = "cuda"
dtype = "float32"

## 1. 字符变编号，目标向后错一位

Tokenizer 按字符首次出现的顺序编号。每段输入有 8 个字符，目标是语料中紧随各位置之后的字符。

In [2]:
with open("data/language.txt", encoding="utf-8") as file:
    text = file.read()
tokenizer = lm.Tokenizer(text)
context_length = 8
data = Stream(tokenizer.encode(text), context_length=context_length, batch_size=2, seed=42, progress=False)
inputs, targets = next(iter(data))
print("词表大小：", tokenizer.size)
print("输入编号：", inputs[0])
visualization.Table(list(zip(tokenizer.decode(inputs[0]), tokenizer.decode(targets[0]))), columns=["输入字符", "目标字符"])

词表大小： 44
输入编号： [41, 11, 19, 17, 18, 42, 43, 12]


## 2. 搭建语言模型

整数编号 → Embedding → 位置表示 → 因果 Transformer → 字符得分。
输出形状是 `(批次, 序列长度, 词表大小)`，交叉熵可直接接收这些得分和整数目标。

In [3]:
model = lm.Model(tokenizer, context_length=context_length, features=8, heads=2, hidden_features=16, seed=42).to(device, dtype=dtype)
optimizer = nn.AdamW(model, learning_rate=0.01)
loss_function = nn.CrossEntropy()
visualization.Network(model)

## 3. 因果注意力：当前位置只能看过去

热图的行是当前预测位置，列是被读取的位置。右上方代表未来字符，权重应为零。

In [4]:
embedded = model.embedding(inputs)
positioned = model.network.layers[0](embedded)
block = model.network.layers[1]
weights = block.attention.weights(block.attention_norm(positioned))[0, 0]
characters = list(tokenizer.decode(inputs[0]))
visualization.Heatmap(weights.tolist(), xlabels=characters, ylabels=characters, title="因果注意力")

## 4. AdamW 训练两步

先 `backward()` 得到梯度，再按整个模型的 L2 范数统一缩放，最后 `step()` 更新。
裁剪限制梯度大小，不改变方向；返回裁剪前的范数，方便查看。AdamW 保存梯度的移动平均，并独立执行权重衰减。
这里只更新两次；固定 seed 便于复现示例，实际使用可省略。

In [5]:
initial_loss = loss_function(model(inputs, train=False), targets).item()
history = [[0, initial_loss]]
print(f"训练前 loss {initial_loss:.6f}")

for step in range(2):
    optimizer.zero_grad()
    loss = loss_function(model(inputs), targets)
    loss.backward()
    gradient_norm = nn.clip_grad_norm(model, max_norm=1.0)
    optimizer.step()
    current_loss = loss_function(model(inputs, train=False), targets).item()
    history.append([step + 1, current_loss])
    print(f"step {step + 1}  loss {current_loss:.6f}  裁剪前梯度范数 {gradient_norm:.6f}")

训练前 loss 4.133514
step 1  loss 3.596162  裁剪前梯度范数 15.055653
step 2  loss 3.422968  裁剪前梯度范数 2.237402


## 5. 对照贪心与随机续写

默认温度为 0，每次选得分最高的字符。正温度按概率抽取：温度越低越偏向高分字符；`top_k=5` 只在五个最高分字符里选。
同一模型与 seed 可以复现采样，省略 seed 则自然随机；两步训练只验证流程，不保证文字通顺。

In [6]:
print("贪心续写：", model.generate("小猫", length=16))
print("随机续写：", model.generate("小猫", length=16, temperature=0.8, top_k=5, seed=42))

贪心续写： 小猫见见见见见见见见见见见见见见见见
随机续写： 小猫一见见见去明里见，见见，，见去明


## 6. KV cache：记住已经算过的 K、V

`generate` 默认启用缓存，无需自己管理。先计算完整开头，之后每步只计算新字符，复用各层已有的 K、V。`use_cache=False` 可以关闭缓存作对照；缓存只在本次续写内使用，不保存到模型文件。

这里开头有 2 个字符，续写 6 个，仍在长度 8 的窗口内。无缓存各步处理 `[2, 3, 4, 5, 6, 7]` 个字符，启用缓存为 `[2, 1, 1, 1, 1, 1]`，每层 K/V 投影处理量从 27 降到 7，输出应相同。

当前模型采用固定长度的可学习位置编码。窗口满后再移入新字符，旧字符的位置和前文都会变化，因此需要重建当前完整窗口的缓存；这时不再有持续的增量收益。前面的 16 字续写也会经过这个过程，保留原来的计算结果。

In [7]:
without_cache = model.generate("小猫", length=6, use_cache=False)
with_cache = model.generate("小猫", length=6)
assert without_cache == with_cache, "缓存前后的续写不一致"
visualization.Table([["关闭", without_cache, 27], ["启用", with_cache, 7]],
                    columns=["KV cache", "续写结果", "每层 K/V 投影处理字符数"])